# 02. Confidence 기반 반복 복원

목표: 한 step에서 모든 mask 위치를 예측하되 confidence가 높은 일부만 확정하는 toy denoising loop를 구현합니다. 실제 neural network 대신 정답 token과 deterministic score를 주는 oracle을 사용합니다.

In [ ]:
MASK = "[MASK]"
target = "병렬 복원 은 여러 위치 를 함께 예측 한다".split()

def oracle_predict(sequence, step):
    predictions = []
    for position, token in enumerate(sequence):
        if token != MASK:
            continue
        confidence = ((position * 37 + step * 19) % 100) / 100
        predictions.append((position, target[position], confidence))
    return predictions

print(oracle_predict([MASK] * len(target), step=1))

## 높은 confidence token부터 확정

매 step 남은 mask의 절반을 확정합니다. 나머지는 다시 mask 상태로 남아 다음 양방향 예측을 기다립니다.

In [ ]:
def denoise(target, max_steps=8):
    sequence = [MASK] * len(target)
    history = []
    for step in range(1, max_steps + 1):
        candidates = sorted(oracle_predict(sequence, step), key=lambda row: row[2], reverse=True)
        if not candidates:
            break
        commit_count = max(1, (len(candidates) + 1) // 2)
        for position, token, _ in candidates[:commit_count]:
            sequence[position] = token
        history.append((step, commit_count, list(sequence)))
    return sequence, history

result, history = denoise(target)
for step, committed, sequence in history:
    print(f"step={step}, committed={committed}:", " ".join(sequence))
assert result == target

## Confidence와 margin의 차이

Top probability가 같아도 2위 후보가 가까우면 margin은 작습니다. Branch가 애매한 위치를 늦게 확정하는 데 margin을 사용할 수 있습니다.

In [ ]:
positions = {
    0: [0.80, 0.10, 0.10],
    1: [0.80, 0.19, 0.01],
    2: [0.62, 0.37, 0.01],
}
for position, probs in positions.items():
    ordered = sorted(probs, reverse=True)
    confidence = ordered[0]
    margin = ordered[0] - ordered[1]
    print(f"position={position}, confidence={confidence:.2f}, margin={margin:.2f}")

## 주의

Toy oracle는 committed token을 틀리지 않습니다. 실제 model은 높은 confidence로 틀릴 수 있고, 너무 이른 확정이 뒤 prediction을 오염시킵니다. 실제 sampler 비교에는 정답률, pass@k, calibration과 latency가 모두 필요합니다.